# LeetCode 827 — Making A Large Island bằng Python

- Link: https://leetcode.com/problems/making-a-large-island/
- Độ khó: **Hard**
- Chủ đề: Lưới 2D, DFS/BFS tô màu thành phần liên thông, Union-Find (DSU), Bảng băm `id → diện tích`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def largestIsland(self, grid: List[List[int]]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Cho một lưới `n × n` gồm `0` (nước) và `1` (đất). Bạn được đổi **tối đa một** ô `0` thành `1`. Hỏi đảo (nhóm ô `1` liên thông 4 hướng) lớn nhất có thể có diện tích bao nhiêu?

Cách ngây thơ: thử đổi từng ô `0`, mỗi lần đếm lại đảo lớn nhất → `O(n⁴)`. Cách thông minh: **đếm mỗi đảo đúng một lần**, gán cho nó một mã số (id) và lưu diện tích; khi thử đổi ô `0`, chỉ cần nhìn 4 ô hàng xóm, lấy các id **khác nhau** và cộng diện tích. Đây là kỹ thuật "tô màu thành phần liên thông + tra bảng" rất hay gặp.

Sau notebook này bạn sẽ:

1. Duyệt lưới 4 hướng bằng DFS (đệ quy và dùng stack) và BFS.
2. Gán **nhãn** cho từng thành phần liên thông và lưu diện tích vào bảng.
3. Hiểu vì sao phải **khử trùng** id hàng xóm bằng `set` khi ghép đảo.
4. Cài Union-Find có nén đường đi + gộp theo kích thước để giải cùng bài.
5. Xử lý biên: lưới toàn `1`, lưới toàn `0`.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho ma trận nhị phân `grid` kích thước `n × n`. Bạn được phép đổi **nhiều nhất một** ô từ `0` thành `1`. Trả về diện tích (số ô) của đảo lớn nhất sau thao tác này. Đảo là nhóm ô `1` nối nhau theo 4 hướng (trên, dưới, trái, phải).

**Ví dụ 1**

```text
Input: grid = [[1,0],[0,1]]
Output: 3
Giải thích: đổi một ô 0 thành 1 nối hai đảo → diện tích 3.
```

**Ví dụ 2**

```text
Input: grid = [[1,1],[1,0]]
Output: 4
```

**Ví dụ 3**

```text
Input: grid = [[1,1],[1,1]]
Output: 4
Giải thích: không có ô 0 nào để đổi, đảo lớn nhất vẫn là 4.
```

**Ràng buộc**

- `n == grid.length == grid[i].length`
- `1 <= n <= 500`
- `grid[i][j]` là `0` hoặc `1`.

In [1]:
from typing import List

EXAMPLES = [
    ([[1, 0], [0, 1]], 3),
    ([[1, 1], [1, 0]], 4),
    ([[1, 1], [1, 1]], 4),
    ([[0]], 1),                       # đổi ô duy nhất
    ([[1]], 1),
    ([[0, 0], [0, 0]], 1),
]
for g, exp in EXAMPLES:
    print(g, "-> kỳ vọng", exp)

[[1, 0], [0, 1]] -> kỳ vọng 3
[[1, 1], [1, 0]] -> kỳ vọng 4
[[1, 1], [1, 1]] -> kỳ vọng 4
[[0]] -> kỳ vọng 1
[[1]] -> kỳ vọng 1
[[0, 0], [0, 0]] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Duyệt 4 hướng trên lưới

Dùng mảng hướng `DIRS = [(1,0),(-1,0),(0,1),(0,-1)]` và kiểm tra `0 <= r < n and 0 <= c < n`.

In [2]:
DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]

def neighbors(r, c, n):
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < n and 0 <= nc < n:
            yield nr, nc

print(list(neighbors(0, 0, 3)), list(neighbors(1, 1, 3)))

[(1, 0), (0, 1)] [(2, 1), (0, 1), (1, 2), (1, 0)]


### 3.2. DFS bằng stack để đếm một thành phần

Đệ quy trong Python bị giới hạn ~1000 tầng; lưới 500×500 có thể có đảo 250 000 ô → dùng **stack** tường minh cho an toàn.

In [3]:
def island_size(grid, r0, c0, seen):
    n = len(grid)
    stack = [(r0, c0)]
    seen.add((r0, c0))
    size = 0
    while stack:
        r, c = stack.pop()
        size += 1
        for nr, nc in neighbors(r, c, n):
            if grid[nr][nc] == 1 and (nr, nc) not in seen:
                seen.add((nr, nc))
                stack.append((nr, nc))
    return size

g = [[1, 1, 0], [0, 1, 0], [1, 0, 1]]
print(island_size(g, 0, 0, set()))

3


### 3.3. BFS với `deque`

Thứ tự thăm khác DFS nhưng **tập ô thăm được** như nhau → cùng diện tích.

In [4]:
from collections import deque

def island_size_bfs(grid, r0, c0, seen):
    n = len(grid)
    q = deque([(r0, c0)])
    seen.add((r0, c0))
    size = 0
    while q:
        r, c = q.popleft()
        size += 1
        for nr, nc in neighbors(r, c, n):
            if grid[nr][nc] == 1 and (nr, nc) not in seen:
                seen.add((nr, nc))
                q.append((nr, nc))
    return size

print(island_size_bfs(g, 0, 0, set()))

3


### 3.4. `set` để khử trùng

Một ô `0` có thể có 2–4 hàng xóm thuộc **cùng một đảo**. Gom id vào `set` rồi mới cộng diện tích.

In [5]:
neighbor_ids = [2, 3, 2, 2]
area = {2: 5, 3: 4}
print("cộng sai (trùng):", 1 + sum(area[i] for i in neighbor_ids))
print("cộng đúng (set):", 1 + sum(area[i] for i in set(neighbor_ids)))

cộng sai (trùng): 20
cộng đúng (set): 10


### 3.5. Union-Find (Disjoint Set Union)

Hai thao tác: `find(x)` tìm đại diện, `union(a, b)` gộp hai tập. Với **nén đường đi** và **gộp theo kích thước**, mỗi thao tác gần như `O(1)`.

In [6]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n))
        self.size = [1] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]   # nén nửa đường
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        return True

d = DSU(5)
d.union(0, 1); d.union(3, 4); d.union(1, 4)
print([d.find(i) for i in range(5)], d.size[d.find(0)])

[0, 0, 2, 0, 0] 4


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Ghép đảo qua một ô nước

Khi đổi ô `(r, c)` từ `0` thành `1`, đảo mới chứa: chính ô đó (1) + **toàn bộ** các đảo chạm vào nó từ 4 phía. Đảo không chạm ô đó thì không đổi.

### 4.2. Tính tay

```text
grid =
1 1 0
0 0 1
1 0 1
```

Tô màu: đảo A = {(0,0),(0,1)} diện tích 2; đảo B = {(1,2),(2,2)} diện tích 2; đảo C = {(2,0)} diện tích 1.

| ô 0 | hàng xóm là đất | id khác nhau | diện tích mới |
|---|---|---|---|
| (0,2) | (0,1)=A, (1,2)=B | {A, B} | 1+2+2 = **5** |
| (1,0) | (0,0)=A, (2,0)=C | {A, C} | 1+2+1 = 4 |
| (1,1) | (0,1)=A, (1,2)=B | {A, B} | 5 |
| (2,1) | (2,0)=C, (2,2)=B | {B, C} | 4 |

→ Đáp án **5**.

### 4.3. Biên

- Không có ô `0` → đáp án = `n²` (toàn bộ lưới là một đảo).
- Toàn `0` → đổi một ô → `1`.

In [7]:
grid_demo = [[1, 1, 0], [0, 0, 1], [1, 0, 1]]
for row in grid_demo:
    print(' '.join('#' if v else '.' for v in row))

# # .
. . #
# . #


## 5. Cách 1 — Brute Force: thử đổi từng ô rồi đếm lại

**Ý tưởng.** Với mỗi ô `0`: đổi thành `1`, tìm đảo lớn nhất toàn lưới (DFS mọi ô), đổi lại. Thêm trường hợp không đổi gì (lưới toàn `1`).

In [8]:
def max_island(grid):
    n = len(grid)
    seen, best = set(), 0
    for r in range(n):
        for c in range(n):
            if grid[r][c] == 1 and (r, c) not in seen:
                best = max(best, island_size(grid, r, c, seen))
    return best

def largest_brute(grid):
    grid = [row[:] for row in grid]          # không sửa input
    n = len(grid)
    best = max_island(grid)                  # trường hợp không đổi ô nào
    for r in range(n):
        for c in range(n):
            if grid[r][c] == 0:
                grid[r][c] = 1
                best = max(best, max_island(grid))
                grid[r][c] = 0
    return best

for g, exp in EXAMPLES:
    assert largest_brute(g) == exp
print("Brute force: OK;", largest_brute(grid_demo))

Brute force: OK; 5


**Độ phức tạp.** Có tới `n²` ô `0`, mỗi lần đếm lại `O(n²)` → `O(n⁴)`. Với `n = 500`: `6.25·10¹⁰` — không thể.

**Nút thắt.** Mỗi lần thử ta đếm lại **toàn bộ** các đảo, dù chúng không đổi. Chỉ đảo chạm vào ô được đổi mới thay đổi!

## 6. Bước đột phá

**Insight.** Diện tích đảo mới khi đổi ô `(r, c)` chỉ phụ thuộc vào **các đảo kề** ô đó. Vậy:

1. **Tiền xử lý một lần:** tô màu mỗi đảo bằng một id riêng (`2, 3, 4, ...` để không lẫn với `0`/`1`), lưu `area[id]`.
2. **Thử từng ô 0:** lấy `set` id của tối đa 4 hàng xóm, kết quả `1 + Σ area[id]`.
3. Đáp án = max của bước 2, và `max(area)` (phòng trường hợp không có ô `0`).

**Invariant.** Sau bước 1, `label[r][c] = id` ⇔ ô `(r, c)` thuộc đảo `id`; `area[id]` = số ô của đảo đó.

**Quy tắc nếu…thì:**
- *Nếu* hai hàng xóm có cùng id *thì* chỉ cộng một lần.
- *Nếu* không có ô `0` nào *thì* đáp án là `n²`.

Tổng độ phức tạp `O(n²)`: mỗi ô được tô màu một lần, mỗi ô `0` xem 4 hàng xóm.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tô màu bằng DFS (stack) + bảng diện tích

Ghi id trực tiếp vào một bản sao của lưới (bắt đầu từ 2).

In [9]:
def largest_dfs(grid):
    n = len(grid)
    lab = [row[:] for row in grid]
    area = {}
    nid = 2
    for r in range(n):
        for c in range(n):
            if lab[r][c] == 1:
                lab[r][c] = nid
                stack, cnt = [(r, c)], 0
                while stack:
                    x, y = stack.pop()
                    cnt += 1
                    for nx, ny in neighbors(x, y, n):
                        if lab[nx][ny] == 1:
                            lab[nx][ny] = nid
                            stack.append((nx, ny))
                area[nid] = cnt
                nid += 1
    best = max(area.values(), default=0)
    for r in range(n):
        for c in range(n):
            if lab[r][c] == 0:
                ids = {lab[nx][ny] for nx, ny in neighbors(r, c, n) if lab[nx][ny] > 1}
                best = max(best, 1 + sum(area[i] for i in ids))
    return best

for g, exp in EXAMPLES:
    assert largest_dfs(g) == exp
print("DFS tô màu: OK;", largest_dfs(grid_demo))

DFS tô màu: OK; 5


Giải thích:

- `lab[r][c] == 1` nghĩa là ô đất **chưa tô**. Khi đưa vào stack ta tô ngay (`= nid`) để không bị thêm hai lần.
- `max(area.values(), default=0)`: nếu lưới toàn `0`, `area` rỗng → 0; khi đó vòng sau cho ít nhất `1`.
- `if lab[nx][ny] > 1`: chỉ lấy ô đã tô (id ≥ 2).

Code nộp LeetCode cho cách 2:

In [10]:
class SolutionDFS:
    def largestIsland(self, grid: List[List[int]]) -> int:
        n = len(grid)
        lab = [row[:] for row in grid]
        area = [0, 0]                       # area[id], id bắt đầu từ 2
        for r in range(n):
            for c in range(n):
                if lab[r][c] != 1:
                    continue
                nid = len(area)
                lab[r][c] = nid
                stack, cnt = [(r, c)], 0
                while stack:
                    x, y = stack.pop()
                    cnt += 1
                    for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                        nx, ny = x + dx, y + dy
                        if 0 <= nx < n and 0 <= ny < n and lab[nx][ny] == 1:
                            lab[nx][ny] = nid
                            stack.append((nx, ny))
                area.append(cnt)
        best = max(area)
        for r in range(n):
            for c in range(n):
                if lab[r][c]:
                    continue
                seen = set()
                for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    nx, ny = r + dx, c + dy
                    if 0 <= nx < n and 0 <= ny < n and lab[nx][ny] > 1:
                        seen.add(lab[nx][ny])
                best = max(best, 1 + sum(area[i] for i in seen))
        return best

for g, exp in EXAMPLES:
    assert SolutionDFS().largestIsland(g) == exp
print("SolutionDFS: OK")

SolutionDFS: OK


**Độ phức tạp cách 2:** `O(n²)` thời gian, `O(n²)` bộ nhớ (bản sao lưới + stack).

### 7.2. Cách 3 — Tô màu bằng BFS

Giống hệt cách 2, chỉ thay stack bằng `deque`. Lợi ích: thứ tự thăm "lan dần" dễ hình dung khi trace.

In [11]:
def largest_bfs(grid):
    n = len(grid)
    lab = [[0] * n for _ in range(n)]
    area = [0]                              # id bắt đầu từ 1 trong mảng lab riêng
    for r in range(n):
        for c in range(n):
            if grid[r][c] == 1 and lab[r][c] == 0:
                nid = len(area)
                lab[r][c] = nid
                q, cnt = deque([(r, c)]), 0
                while q:
                    x, y = q.popleft()
                    cnt += 1
                    for nx, ny in neighbors(x, y, n):
                        if grid[nx][ny] == 1 and lab[nx][ny] == 0:
                            lab[nx][ny] = nid
                            q.append((nx, ny))
                area.append(cnt)
    best = max(area)
    for r in range(n):
        for c in range(n):
            if grid[r][c] == 0:
                ids = {lab[nx][ny] for nx, ny in neighbors(r, c, n)} - {0}
                best = max(best, 1 + sum(area[i] for i in ids))
    return best

for g, exp in EXAMPLES:
    assert largest_bfs(g) == exp
print("BFS tô màu: OK")

BFS tô màu: OK


Ở đây ta **không sửa** `grid` mà dùng mảng `lab` riêng (id ≥ 1, 0 = chưa tô / nước) — thói quen tốt khi đề không cho phép thay đổi input.

Code nộp LeetCode cho cách 3:

In [12]:
class SolutionBFS:
    def largestIsland(self, grid: List[List[int]]) -> int:
        n = len(grid)
        lab = [[0] * n for _ in range(n)]
        area = [0]
        D = ((1, 0), (-1, 0), (0, 1), (0, -1))
        for r in range(n):
            for c in range(n):
                if grid[r][c] == 1 and lab[r][c] == 0:
                    nid = len(area)
                    lab[r][c] = nid
                    q, cnt = deque([(r, c)]), 0
                    while q:
                        x, y = q.popleft()
                        cnt += 1
                        for dx, dy in D:
                            nx, ny = x + dx, y + dy
                            if 0 <= nx < n and 0 <= ny < n and grid[nx][ny] == 1 and lab[nx][ny] == 0:
                                lab[nx][ny] = nid
                                q.append((nx, ny))
                    area.append(cnt)
        best = max(area)
        for r in range(n):
            for c in range(n):
                if grid[r][c] == 0:
                    ids = set()
                    for dx, dy in D:
                        nx, ny = r + dx, c + dy
                        if 0 <= nx < n and 0 <= ny < n and lab[nx][ny]:
                            ids.add(lab[nx][ny])
                    best = max(best, 1 + sum(area[i] for i in ids))
        return best

for g, exp in EXAMPLES:
    assert SolutionBFS().largestIsland(g) == exp
print("SolutionBFS: OK")

SolutionBFS: OK


**Độ phức tạp cách 3:** `O(n²)` / `O(n²)`.

### 7.3. Cách 4 — Union-Find

Đánh số ô `(r, c)` thành `r·n + c`. Với mỗi ô đất, `union` với ô đất bên phải và bên dưới (đủ để phủ mọi cạnh kề). Sau đó đại diện `find(x)` đóng vai trò id, `size[root]` là diện tích.

In [13]:
def largest_dsu(grid):
    n = len(grid)
    dsu = DSU(n * n)
    for r in range(n):
        for c in range(n):
            if grid[r][c] == 1:
                if r + 1 < n and grid[r + 1][c] == 1:
                    dsu.union(r * n + c, (r + 1) * n + c)
                if c + 1 < n and grid[r][c + 1] == 1:
                    dsu.union(r * n + c, r * n + c + 1)
    best = 0
    has_zero = False
    for r in range(n):
        for c in range(n):
            if grid[r][c] == 1:
                best = max(best, dsu.size[dsu.find(r * n + c)])
            else:
                has_zero = True
                roots = {dsu.find(nr * n + nc) for nr, nc in neighbors(r, c, n) if grid[nr][nc] == 1}
                best = max(best, 1 + sum(dsu.size[x] for x in roots))
    return best

for g, exp in EXAMPLES:
    assert largest_dsu(g) == exp
print("Union-Find: OK")

Union-Find: OK


Lưu ý: chỉ đọc `size[root]` của **gốc**; `size` của nút không phải gốc là giá trị cũ, vô nghĩa.

Code nộp LeetCode cho cách 4:

In [14]:
class SolutionDSU:
    def largestIsland(self, grid: List[List[int]]) -> int:
        n = len(grid)
        parent = list(range(n * n))
        size = [1] * (n * n)

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        def union(a, b):
            ra, rb = find(a), find(b)
            if ra == rb:
                return
            if size[ra] < size[rb]:
                ra, rb = rb, ra
            parent[rb] = ra
            size[ra] += size[rb]

        for r in range(n):
            for c in range(n):
                if grid[r][c]:
                    if r + 1 < n and grid[r + 1][c]:
                        union(r * n + c, (r + 1) * n + c)
                    if c + 1 < n and grid[r][c + 1]:
                        union(r * n + c, r * n + c + 1)
        best = 0
        for r in range(n):
            for c in range(n):
                if grid[r][c]:
                    best = max(best, size[find(r * n + c)])
                    continue
                roots = set()
                for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                    if 0 <= nr < n and 0 <= nc < n and grid[nr][nc]:
                        roots.add(find(nr * n + nc))
                best = max(best, 1 + sum(size[x] for x in roots))
        return best

for g, exp in EXAMPLES:
    assert SolutionDSU().largestIsland(g) == exp
print("SolutionDSU: OK")

SolutionDSU: OK


**Độ phức tạp cách 4:** `O(n² · α(n²))` ≈ `O(n²)`, bộ nhớ `O(n²)`. Ưu điểm: nếu lưới được cập nhật dần (thêm đất từng ô — bài 305 Number of Islands II), DSU xử lý trực tuyến tốt hơn DFS.

## 8. TRACE từng bước

Trace cách 2 trên `grid_demo`: in mảng nhãn sau khi tô, bảng diện tích, rồi từng ô `0` với id hàng xóm.

In [15]:
def trace_largest(grid):
    n = len(grid)
    lab = [row[:] for row in grid]
    area = {}
    nid = 2
    for r in range(n):
        for c in range(n):
            if lab[r][c] == 1:
                lab[r][c] = nid
                stack, cells = [(r, c)], []
                while stack:
                    x, y = stack.pop()
                    cells.append((x, y))
                    for nx, ny in neighbors(x, y, n):
                        if lab[nx][ny] == 1:
                            lab[nx][ny] = nid
                            stack.append((nx, ny))
                area[nid] = len(cells)
                print(f"đảo id={nid}: bắt đầu tại {(r, c)}, các ô {sorted(cells)}, diện tích {len(cells)}")
                nid += 1
    print("Nhãn sau khi tô:")
    for row in lab:
        print("   ", row)
    best = max(area.values(), default=0)
    print("best ban đầu (không đổi ô nào) =", best)
    for r in range(n):
        for c in range(n):
            if lab[r][c] == 0:
                ids = {lab[nx][ny] for nx, ny in neighbors(r, c, n) if lab[nx][ny] > 1}
                val = 1 + sum(area[i] for i in ids)
                best = max(best, val)
                print(f"ô 0 {(r, c)}: id hàng xóm {sorted(ids)} -> 1 + {[area[i] for i in sorted(ids)]} = {val}; best = {best}")
    return best

trace_largest(grid_demo)

đảo id=2: bắt đầu tại (0, 0), các ô [(0, 0), (0, 1)], diện tích 2
đảo id=3: bắt đầu tại (1, 2), các ô [(1, 2), (2, 2)], diện tích 2
đảo id=4: bắt đầu tại (2, 0), các ô [(2, 0)], diện tích 1
Nhãn sau khi tô:
    [2, 2, 0]
    [0, 0, 3]
    [4, 0, 3]
best ban đầu (không đổi ô nào) = 2
ô 0 (0, 2): id hàng xóm [2, 3] -> 1 + [2, 2] = 5; best = 5
ô 0 (1, 0): id hàng xóm [2, 4] -> 1 + [2, 1] = 4; best = 5
ô 0 (1, 1): id hàng xóm [2, 3] -> 1 + [2, 2] = 5; best = 5
ô 0 (2, 1): id hàng xóm [3, 4] -> 1 + [2, 1] = 4; best = 5


5

Bảng trạng thái:

| bước | ô | ids | giá trị | best |
|---|---|---|---|---|
| khởi tạo | — | — | max(area) = 2 | 2 |
| 1 | (0,2) | {2, 3} | 5 | 5 |
| 2 | (1,0) | {2, 4} | 4 | 5 |
| 3 | (1,1) | {2, 3} | 5 | 5 |
| 4 | (2,1) | {3, 4} | 4 | 5 |

(id `3` là đảo bắt đầu ở `(1,2)`, id `4` là đảo `(2,0)` — thứ tự id theo thứ tự quét hàng.)

Trace thêm trường hợp một ô `0` bị **cùng một đảo bao quanh** — `set` giúp không cộng trùng:

In [16]:
ring = [[1, 1, 1],
        [1, 0, 1],
        [1, 1, 1]]
trace_largest(ring)

đảo id=2: bắt đầu tại (0, 0), các ô [(0, 0), (0, 1), (0, 2), (1, 0), (1, 2), (2, 0), (2, 1), (2, 2)], diện tích 8
Nhãn sau khi tô:
    [2, 2, 2]
    [2, 0, 2]
    [2, 2, 2]
best ban đầu (không đổi ô nào) = 8
ô 0 (1, 1): id hàng xóm [2] -> 1 + [8] = 9; best = 9


9

## 9. Code nộp LeetCode

Chọn cách 2 (DFS stack + bảng diện tích): đơn giản, không đệ quy, `O(n²)`.

In [17]:
from typing import List

class Solution:
    def largestIsland(self, grid: List[List[int]]) -> int:
        n = len(grid)
        lab = [row[:] for row in grid]          # không sửa input
        area = [0, 0]                           # area[id]; id đảo bắt đầu từ 2
        D = ((1, 0), (-1, 0), (0, 1), (0, -1))
        # Bước 1: tô màu từng đảo, lưu diện tích
        for r in range(n):
            for c in range(n):
                if lab[r][c] != 1:
                    continue
                nid = len(area)
                lab[r][c] = nid
                stack, cnt = [(r, c)], 0
                while stack:
                    x, y = stack.pop()
                    cnt += 1
                    for dx, dy in D:
                        nx, ny = x + dx, y + dy
                        if 0 <= nx < n and 0 <= ny < n and lab[nx][ny] == 1:
                            lab[nx][ny] = nid
                            stack.append((nx, ny))
                area.append(cnt)
        # Bước 2: thử đổi từng ô 0
        best = max(area)                        # = n*n nếu không có ô 0
        for r in range(n):
            for c in range(n):
                if lab[r][c] != 0:
                    continue
                ids = set()
                for dx, dy in D:
                    nx, ny = r + dx, c + dy
                    if 0 <= nx < n and 0 <= ny < n and lab[nx][ny] > 1:
                        ids.add(lab[nx][ny])
                best = max(best, 1 + sum(area[i] for i in ids))
        return best

for g, exp in EXAMPLES:
    assert Solution().largestIsland(g) == exp
print("Solution: OK")

Solution: OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1.** Sau bước tô màu, hai ô đất có cùng id ⇔ chúng cùng đảo. (DFS từ một ô đi được tới đúng các ô liên thông với nó; ô đã tô không bị tô lại nên mỗi ô thuộc đúng một id.) `area[id]` đếm đúng số ô bị tô id đó.

**Bổ đề 2.** Đổi ô `x = (r, c)` từ `0` thành `1`. Đảo chứa `x` trong lưới mới = `{x} ∪` hợp các đảo cũ có ô kề `x`. Lý do: một đường đi 4 hướng từ `x` tới ô đất `y` phải rời `x` qua một hàng xóm `h` là đất; phần còn lại của đường đi không qua `x` (có thể bỏ chu trình) nên nằm hoàn toàn trong đảo cũ của `h`. Ngược lại mọi ô của các đảo kề đều nối được với `x`.

Các đảo cũ này rời nhau, nên diện tích = `1 + Σ area[id]` với `id` chạy qua **tập** (không lặp) id hàng xóm.

**Bổ đề 3.** Các đảo không chứa `x` không thay đổi và có diện tích `≤ max(area)`; mà đảo chứa `x` có diện tích ≥ diện tích mỗi đảo kề. Vậy đáp án = `max( max(area), max_x (1 + Σ area) )`. Khi có ít nhất một ô `0`, giá trị thứ hai luôn ≥ đảo lớn nhất kề nó; còn đảo lớn nhất không kề ô `0` nào chỉ xảy ra khi nó chiếm cả lưới — được `max(area)` lo.

**DSU (cách 4)** tính cùng phân hoạch thành phần liên thông: `union` theo mọi cạnh kề giữa hai ô đất (phải và dưới là đủ vì cạnh vô hướng).

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(n⁴)` | `O(n²)` |
| 2. DFS tô màu | `O(n²)` | `O(n²)` |
| 3. BFS tô màu | `O(n²)` | `O(n²)` |
| 4. Union-Find | `O(n² α(n²))` | `O(n²)` |

In [18]:
import random, time

def random_grid(n, p, rnd):
    return [[1 if rnd.random() < p else 0 for _ in range(n)] for _ in range(n)]

rnd = random.Random(827)
big = random_grid(500, 0.55, rnd)
for name, fn in [("dfs", largest_dfs), ("bfs", largest_bfs), ("dsu", largest_dsu),
                 ("Solution", lambda g: Solution().largestIsland(g))]:
    t0 = time.perf_counter()
    res = fn(big)
    print(f"{name:9}: {res}  ({(time.perf_counter() - t0) * 1000:.0f} ms)")

dfs      : 2533  (368 ms)


bfs      : 2533  (416 ms)


dsu      : 2533  (325 ms)


Solution : 2533  (241 ms)


## 12. Bộ test

In [19]:
methods = {
    "brute": largest_brute,
    "dfs": largest_dfs,
    "bfs": largest_bfs,
    "dsu": largest_dsu,
    "SolutionDFS": lambda g: SolutionDFS().largestIsland(g),
    "SolutionBFS": lambda g: SolutionBFS().largestIsland(g),
    "SolutionDSU": lambda g: SolutionDSU().largestIsland(g),
    "Solution": lambda g: Solution().largestIsland(g),
}

# 12.1 Ví dụ + edge cases
edge = EXAMPLES + [
    (grid_demo, 5),
    (ring, 9),
    ([[1, 0, 1], [0, 0, 0], [1, 0, 1]], 3),
    ([[0, 1, 0], [1, 0, 1], [0, 1, 0]], 5),     # ô giữa nối 4 đảo
    ([[1] * 4 for _ in range(4)], 16),
    ([[0] * 4 for _ in range(4)], 1),
]
for name, fn in methods.items():
    for g, exp in edge:
        assert fn(g) == exp, (name, g, fn(g), exp)
print("Ví dụ + edge cases: OK")

# 12.2 Ngẫu nhiên có seed: đối chiếu brute force
for seed in range(500):
    r_ = random.Random(seed)
    n = r_.randint(1, 6)
    g = random_grid(n, r_.random(), r_)
    ref = largest_brute(g)
    for name, fn in methods.items():
        assert fn(g) == ref, (name, g)
print("500 lưới ngẫu nhiên: khớp brute force")

# 12.3 Input không bị sửa
g = [[1, 0], [0, 1]]
for name, fn in methods.items():
    fn(g)
assert g == [[1, 0], [0, 1]]
print("Input giữ nguyên")

Ví dụ + edge cases: OK
500 lưới ngẫu nhiên: khớp brute force
Input giữ nguyên


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Cộng trùng một đảo nhiều lần

**Code sai:**

In [20]:
def wrong_no_set(grid):
    n = len(grid)
    lab = [row[:] for row in grid]
    area, nid = {}, 2
    for r in range(n):
        for c in range(n):
            if lab[r][c] == 1:
                lab[r][c] = nid; st, cnt = [(r, c)], 0
                while st:
                    x, y = st.pop(); cnt += 1
                    for nx, ny in neighbors(x, y, n):
                        if lab[nx][ny] == 1:
                            lab[nx][ny] = nid; st.append((nx, ny))
                area[nid] = cnt; nid += 1
    best = max(area.values(), default=0)
    for r in range(n):
        for c in range(n):
            if lab[r][c] == 0:
                ids = [lab[nx][ny] for nx, ny in neighbors(r, c, n) if lab[nx][ny] > 1]   # SAI: list
                best = max(best, 1 + sum(area[i] for i in ids))
    return best

print("sai:", wrong_no_set(ring), " đúng:", Solution().largestIsland(ring))

sai: 33  đúng: 9


**Vì sao sai:** ô giữa có 4 hàng xóm cùng thuộc đảo vòng (8 ô) → cộng `4 × 8`. Kết quả vượt cả `n²`!

**Code đúng:** gom id vào `set`.

### Lỗi 2 — Quên trường hợp không có ô 0

**Code sai:**

In [21]:
def wrong_all_ones(grid):
    n = len(grid)
    lab = [row[:] for row in grid]
    area, nid = {}, 2
    for r in range(n):
        for c in range(n):
            if lab[r][c] == 1:
                lab[r][c] = nid; st, cnt = [(r, c)], 0
                while st:
                    x, y = st.pop(); cnt += 1
                    for nx, ny in neighbors(x, y, n):
                        if lab[nx][ny] == 1:
                            lab[nx][ny] = nid; st.append((nx, ny))
                area[nid] = cnt; nid += 1
    best = 0                                              # SAI: không khởi tạo bằng max(area)
    for r in range(n):
        for c in range(n):
            if lab[r][c] == 0:
                ids = {lab[nx][ny] for nx, ny in neighbors(r, c, n) if lab[nx][ny] > 1}
                best = max(best, 1 + sum(area[i] for i in ids))
    return best

print("sai:", wrong_all_ones([[1, 1], [1, 1]]), " đúng:", Solution().largestIsland([[1, 1], [1, 1]]))

sai: 0  đúng: 4


**Vì sao sai:** không có ô `0` → vòng thử không chạy → trả `0`. **Code đúng:** `best = max(area)` (hoặc `n*n` nếu không có ô 0).

### Lỗi 3 — Dùng nhãn `1` cho đảo đầu tiên

Nếu đánh id từ `1`, ô đã tô và ô đất **chưa tô** đều mang giá trị `1` → DFS không phân biệt được, đảo đầu tiên bị đếm sai hoặc trùng với đảo khác.

In [22]:
def wrong_id_from_one(grid):
    """Đánh id từ 1: ô đã tô và ô đất chưa tô đều mang giá trị 1."""
    n = len(grid)
    lab = [row[:] for row in grid]
    area, nid = {}, 1                                      # SAI: id đầu tiên trùng giá trị "đất chưa tô"
    for r in range(n):
        for c in range(n):
            if lab[r][c] == 1:                             # ô đã tô id 1 cũng thoả điều kiện này!
                lab[r][c] = nid
                st, cnt, visited = [(r, c)], 0, {(r, c)}
                while st:
                    x, y = st.pop(); cnt += 1
                    for nx, ny in neighbors(x, y, n):
                        if lab[nx][ny] in (1, nid) and (nx, ny) not in visited:   # cần visited phụ để khỏi lặp vô hạn
                            visited.add((nx, ny)); lab[nx][ny] = nid; st.append((nx, ny))
                area[nid] = cnt
                nid += 1
    return area

print("bảng diện tích sai:", wrong_id_from_one([[1, 0, 1], [1, 0, 1], [0, 0, 1]]))

bảng diện tích sai: {1: 2, 2: 3, 3: 2}


**Vì sao sai:** điều kiện `lab == 1` không còn nghĩa "chưa thăm". Ở demo, lưới có 2 đảo (diện tích 2 và 3) nhưng bảng diện tích có tới 3 mục: ô `(1,0)` đã được tô id `1` nhưng vẫn mang giá trị `1` nên bị coi là "đất chưa tô" và đếm thành một đảo mới. Nếu bỏ tập `visited` phụ thì DFS còn lặp vô hạn. **Code đúng:** id bắt đầu từ `2`, hoặc dùng mảng nhãn riêng như cách 3.

### Lỗi 4 — DFS đệ quy trên lưới lớn

`def dfs(r, c): ... dfs(nr, nc)` với đảo 250 000 ô → `RecursionError`. Dùng stack/BFS, hoặc tăng `sys.setrecursionlimit` (vẫn có thể tràn stack C).

## 14. Mẹo & nhận dạng pattern

- **"Đổi tối đa một ô / thêm một cạnh, hỏi thành phần lớn nhất"** → tiền xử lý thành phần liên thông (id + kích thước), rồi thử từng vị trí chỉ nhìn **hàng xóm**.
- **Khử trùng id** bằng `set` mỗi khi gộp nhiều thành phần qua một điểm.
- **Tô màu trực tiếp vào lưới** (id ≥ 2) tiết kiệm bộ nhớ `visited`.
- **DSU** thích hợp khi lưới thay đổi dần (thêm đất), DFS/BFS khi lưới cố định.
- Bài liên quan: **200 Number of Islands**, **695 Max Area of Island**, **305 Number of Islands II**, **1254 Number of Closed Islands**, **934 Shortest Bridge**.

## 15. Biến thể

1. **Được đổi tối đa `k` ô**: khó hơn nhiều (NP-khó tổng quát); với `k` nhỏ có thể thử tổ hợp.
2. **Liên thông 8 hướng**: đổi mảng hướng thành 8 phần tử.
3. **Trả về vị trí ô nên đổi**: lưu `argmax` trong vòng thử.
4. **Lưới chữ nhật `m × n`**: thay `n` bằng `m`, `n` riêng.
5. **Đổi một ô `1` thành `0` làm đảo lớn nhất nhỏ nhất** (điểm khớp nối): liên quan cầu/đỉnh khớp (Tarjan).

Minh hoạ biến thể 3 + 2 — trả về vị trí và hỗ trợ 8 hướng:

In [23]:
def best_flip(grid, eight=False):
    n = len(grid)
    D = [(1, 0), (-1, 0), (0, 1), (0, -1)]
    if eight:
        D += [(1, 1), (1, -1), (-1, 1), (-1, -1)]
    lab = [row[:] for row in grid]
    area = [0, 0]
    for r in range(n):
        for c in range(n):
            if lab[r][c] == 1:
                nid = len(area); lab[r][c] = nid; st, cnt = [(r, c)], 0
                while st:
                    x, y = st.pop(); cnt += 1
                    for dx, dy in D:
                        nx, ny = x + dx, y + dy
                        if 0 <= nx < n and 0 <= ny < n and lab[nx][ny] == 1:
                            lab[nx][ny] = nid; st.append((nx, ny))
                area.append(cnt)
    best, pos = max(area), None
    for r in range(n):
        for c in range(n):
            if lab[r][c] == 0:
                ids = {lab[r + dx][c + dy] for dx, dy in D
                       if 0 <= r + dx < n and 0 <= c + dy < n and lab[r + dx][c + dy] > 1}
                v = 1 + sum(area[i] for i in ids)
                if v > best:
                    best, pos = v, (r, c)
    return best, pos

print(best_flip(grid_demo))
print(best_flip([[1, 0, 0], [0, 0, 0], [0, 0, 1]], eight=True))

(5, (0, 2))
(3, (1, 1))


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | đổi từng ô, đếm lại toàn lưới | `O(n⁴)` | `O(n²)` | đối chiếu, `n ≤ 10` |
| 2. DFS tô màu | id + diện tích, thử ô 0 qua hàng xóm | `O(n²)` | `O(n²)` | **nộp bài** |
| 3. BFS tô màu | như 2, dùng queue, mảng nhãn riêng | `O(n²)` | `O(n²)` | khi không được sửa input |
| 4. Union-Find | gộp ô kề, `size[root]` | `O(n² α)` | `O(n²)` | lưới cập nhật dần |

## 17. Bài tập

**Bài 1 (đoán output).** `grid = [[1,0,1],[0,0,0],[1,0,1]]` cho đáp án bao nhiêu? `[[0,1,0],[1,0,1],[0,1,0]]`?

**Bài 2 (tính tay).** Với lưới dưới, tô màu (id theo thứ tự quét hàng từ 2) và tính đáp án:

```text
1 1 0 0
1 0 0 1
0 0 1 1
1 0 0 0
```

**Bài 3 (điền TODO).** Viết `count_islands(grid)` đếm số đảo bằng DSU.

**Bài 4 (tự viết lại).** Viết lại `Solution` cho lưới chữ nhật `m × n` (đề gốc là vuông).

**Bài 5 (mini challenge).** Mỗi ô có thể đổi với chi phí 1; bạn có ngân sách đổi **đúng 2** ô. Viết brute force thông minh: thử mọi cặp ô `0` nhưng tính diện tích bằng bảng id + BFS nhỏ quanh hai ô.

In [24]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def count_islands_todo(grid):
    # TODO: DSU trên các ô đất; đếm số gốc
    pass

# Bài 4 — TODO
def largest_rect_todo(grid):
    # TODO: m = len(grid), n = len(grid[0]); giống Solution nhưng biên riêng
    pass

# Bài 5 — TODO
def largest_two_flips_todo(grid):
    # TODO: nếu hai ô kề nhau thì gộp; ngược lại hai cụm hàng xóm có thể chung id
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: 4 đảo diện tích 1 ở 4 góc.
- Mức 2: mỗi ô 0 ở cạnh chạm tối đa 2 góc; ô giữa không chạm góc nào.
- Mức 3: `3`; lưới thứ hai: ô giữa chạm 4 đảo → `5`.

**Bài 2.**
- Mức 1: đảo 2 = {(0,0),(0,1),(1,0)} diện tích 3; đảo 3 = {(1,3),(2,2),(2,3)} diện tích 3; đảo 4 = {(3,0)} diện tích 1.
- Mức 2: lần lượt xét từng ô 0 và tập id hàng xóm: (0,2)→{2}, (1,1)→{2}, (1,2)→{3} (hai lần, nhờ set chỉ cộng một lần), (2,0)→{2, 4}, (3,1)→{4}, ...
- Mức 3: ô (2,0) cho `1 + 3 + 1 = 5`, các ô khác cho tối đa `1 + 3 = 4` → đáp án **5**.

**Bài 3.**
- Mức 1: `DSU(n*n)`.
- Mức 2: union với phải/dưới.
- Mức 3: đếm `x` là ô đất và `find(x) == x`.

**Bài 4.**
- Mức 1: `m, n = len(grid), len(grid[0])`.
- Mức 2: điều kiện biên `0 <= nx < m and 0 <= ny < n`.
- Mức 3: test với lưới `1 × k` và `k × 1`.

**Bài 5.**
- Mức 1: dùng brute force đổi 2 ô (`O(n⁶)`) làm chuẩn.
- Mức 2: với cặp ô, tập id = hợp id hàng xóm của cả hai **nếu** hai ô kề nhau (hoặc cùng chạm một đảo).
- Mức 3: không kề nhau và không chung đảo → hai đảo mới riêng, lấy max.

## 19. Đáp án tham khảo

In [25]:
# Bài 1
assert Solution().largestIsland([[1, 0, 1], [0, 0, 0], [1, 0, 1]]) == 3
assert Solution().largestIsland([[0, 1, 0], [1, 0, 1], [0, 1, 0]]) == 5

# Bài 2
g2 = [[1, 1, 0, 0], [1, 0, 0, 1], [0, 0, 1, 1], [1, 0, 0, 0]]
assert Solution().largestIsland(g2) == largest_brute(g2) == 5

# Bài 3
def count_islands(grid):
    n = len(grid)
    d = DSU(n * n)
    for r in range(n):
        for c in range(n):
            if grid[r][c]:
                if r + 1 < n and grid[r + 1][c]:
                    d.union(r * n + c, (r + 1) * n + c)
                if c + 1 < n and grid[r][c + 1]:
                    d.union(r * n + c, r * n + c + 1)
    return sum(1 for r in range(n) for c in range(n) if grid[r][c] and d.find(r * n + c) == r * n + c)

assert count_islands(g2) == 3
assert count_islands([[1, 0, 1], [0, 0, 0], [1, 0, 1]]) == 4

# Bài 4
def largest_rect(grid):
    m, n = len(grid), len(grid[0])
    lab = [row[:] for row in grid]
    area = [0, 0]
    D = ((1, 0), (-1, 0), (0, 1), (0, -1))
    for r in range(m):
        for c in range(n):
            if lab[r][c] == 1:
                nid = len(area); lab[r][c] = nid; st, cnt = [(r, c)], 0
                while st:
                    x, y = st.pop(); cnt += 1
                    for dx, dy in D:
                        nx, ny = x + dx, y + dy
                        if 0 <= nx < m and 0 <= ny < n and lab[nx][ny] == 1:
                            lab[nx][ny] = nid; st.append((nx, ny))
                area.append(cnt)
    best = max(area)
    for r in range(m):
        for c in range(n):
            if lab[r][c] == 0:
                ids = {lab[r + dx][c + dy] for dx, dy in D
                       if 0 <= r + dx < m and 0 <= c + dy < n and lab[r + dx][c + dy] > 1}
                best = max(best, 1 + sum(area[i] for i in ids))
    return best

assert largest_rect([[1, 0, 1, 1, 0, 1]]) == 4
assert largest_rect([[1], [0], [1]]) == 3
for seed in range(100):
    r_ = random.Random(seed)
    g = random_grid(r_.randint(1, 5), r_.random(), r_)
    assert largest_rect(g) == Solution().largestIsland(g)

# Bài 5 (brute force làm chuẩn cho n nhỏ)
def largest_two_flips_brute(grid):
    n = len(grid)
    zeros = [(r, c) for r in range(n) for c in range(n) if grid[r][c] == 0]
    g = [row[:] for row in grid]
    if len(zeros) <= 2:
        return n * n
    best = 0
    for i in range(len(zeros)):
        for j in range(i + 1, len(zeros)):
            (a, b), (c, d) = zeros[i], zeros[j]
            g[a][b] = g[c][d] = 1
            best = max(best, max_island(g))
            g[a][b] = g[c][d] = 0
    return best

print("đổi 2 ô:", largest_two_flips_brute([[1, 0, 0, 1], [0, 0, 0, 0], [0, 0, 0, 0], [1, 0, 0, 1]]))
print("Tất cả đáp án đúng")

đổi 2 ô: 4
Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (tô màu + tra bảng):**

```python
lab = [row[:] for row in grid]; area = [0, 0]
for r in range(n):
    for c in range(n):
        if lab[r][c] == 1:
            nid = len(area); lab[r][c] = nid; st, cnt = [(r, c)], 0
            while st:
                x, y = st.pop(); cnt += 1
                for nx, ny in nbrs(x, y):
                    if lab[nx][ny] == 1: lab[nx][ny] = nid; st.append((nx, ny))
            area.append(cnt)
best = max(area)
for each zero (r, c):
    ids = {lab[v] for v in nbrs(r, c) if lab[v] > 1}
    best = max(best, 1 + sum(area[i] for i in ids))
```

**Checklist trước khi submit:**

- [ ] id đảo bắt đầu từ **2** (hoặc mảng nhãn riêng).
- [ ] Gom id hàng xóm bằng **`set`**.
- [ ] `best` khởi tạo bằng `max(area)` → đúng khi lưới toàn `1`.
- [ ] Lưới toàn `0` → `1`.
- [ ] DFS bằng **stack**, không đệ quy sâu.